In [21]:
import pandas as pd
from pathlib import Path


In [22]:
path_securite = Path("../../../data/silver/RAYAN securite_data_silver.csv")
df_securite = pd.read_csv(path_securite, sep=";", dtype=str)

print(df_securite.shape)
display(df_securite.head())


(37125, 15)


,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux,timestamp_processing,bronze_load_date
0,69001,2016-01-01,Violences physiques intrafamiliales,Victime,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
1,69001,2016-01-01,Violences physiques hors cadre familial,Victime,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
2,69001,2016-01-01,Violences sexuelles,Victime,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
3,69001,2016-01-01,Vols avec armes,Infraction,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
4,69001,2016-01-01,Vols violents sans arme,Infraction,1.5816327,0.3170966,ndiff,358,2016-01-01,163,2016-01-01,1.5816327,0.3170966,2026-03-23 14:13:09.597346,2026-03-23


In [23]:
# Clean column names
df_securite.columns = df_securite.columns.str.strip()

print(df_securite.columns.tolist())

['CODGEO_2025', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux', 'timestamp_processing', 'bronze_load_date']


In [24]:
# Trim string values in all columns
for col in df_securite.columns:
    df_securite[col] = df_securite[col].astype(str).str.strip()

display(df_securite.head())


,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux,timestamp_processing,bronze_load_date
0,69001,2016-01-01,Violences physiques intrafamiliales,Victime,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
1,69001,2016-01-01,Violences physiques hors cadre familial,Victime,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
2,69001,2016-01-01,Violences sexuelles,Victime,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
3,69001,2016-01-01,Vols avec armes,Infraction,0.0,0.0,diff,358,2016-01-01,163,2016-01-01,0.0,0.0,2026-03-23 14:13:09.597346,2026-03-23
4,69001,2016-01-01,Vols violents sans arme,Infraction,1.5816327,0.3170966,ndiff,358,2016-01-01,163,2016-01-01,1.5816327,0.3170966,2026-03-23 14:13:09.597346,2026-03-23


In [25]:
# Rename columns to a clean technical schema
df_securite = df_securite.rename(columns={
    "CODGEO_2025": "code_insee_commune",
    "annee": "annee",
    "indicateur": "indicateur",
    "unite_de_compte": "unite_de_compte",
    "nombre": "nombre",
    "taux_pour_mille": "taux_pour_mille",
    "est_diffuse": "est_diffuse",
    "insee_pop": "insee_pop",
    "insee_pop_millesime": "insee_pop_millesime",
    "insee_log": "insee_log",
    "insee_log_millesime": "insee_log_millesime",
    "complement_info_nombre": "complement_info_nombre",
    "complement_info_taux": "complement_info_taux",
    "timestamp_processing": "source_dataset_timestamp",
    "bronze_load_date": "date_refresh"
})

print(df_securite.columns.tolist())


['code_insee_commune', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux', 'source_dataset_timestamp', 'date_refresh']


In [26]:
# Split the 5-digit INSEE commune code into department and commune codes
df_securite["code_insee_commune"] = df_securite["code_insee_commune"].str.zfill(5)
df_securite["code_departement"] = df_securite["code_insee_commune"].str[:2]
df_securite["code_commune"] = df_securite["code_insee_commune"].str[2:]

display(df_securite[["code_insee_commune", "code_departement", "code_commune"]].head())


,code_insee_commune,code_departement,code_commune
0,69001,69,001
1,69001,69,001
2,69001,69,001
3,69001,69,001
4,69001,69,001


In [27]:
# Convert year-like dates into an integer year
df_securite["annee"] = pd.to_datetime(df_securite["annee"], errors="coerce").dt.year.astype("Int64")
df_securite["insee_pop_millesime"] = pd.to_datetime(df_securite["insee_pop_millesime"], errors="coerce")
df_securite["insee_log_millesime"] = pd.to_datetime(df_securite["insee_log_millesime"], errors="coerce")
df_securite["source_dataset_timestamp"] = pd.to_datetime(df_securite["source_dataset_timestamp"], errors="coerce")
df_securite["date_refresh"] = pd.to_datetime(df_securite["date_refresh"], errors="coerce")

display(df_securite[["annee", "insee_pop_millesime", "insee_log_millesime", "source_dataset_timestamp", "date_refresh"]].head())


,annee,insee_pop_millesime,insee_log_millesime,source_dataset_timestamp,date_refresh
0,2016,2016-01-01,2016-01-01,2026-03-23 14:13:09.597346,2026-03-23
1,2016,2016-01-01,2016-01-01,2026-03-23 14:13:09.597346,2026-03-23
2,2016,2016-01-01,2016-01-01,2026-03-23 14:13:09.597346,2026-03-23
3,2016,2016-01-01,2016-01-01,2026-03-23 14:13:09.597346,2026-03-23
4,2016,2016-01-01,2016-01-01,2026-03-23 14:13:09.597346,2026-03-23


In [28]:
# Convert numeric columns
numeric_cols = [
    "nombre",
    "taux_pour_mille",
    "insee_pop",
    "insee_log",
    "complement_info_nombre",
    "complement_info_taux"
]

for col in numeric_cols:
    df_securite[col] = pd.to_numeric(df_securite[col], errors="coerce")

display(df_securite[numeric_cols].dtypes)


nombre                    float64
taux_pour_mille           float64
insee_pop                   int64
insee_log                   int64
complement_info_nombre    float64
complement_info_taux      float64
dtype: object

In [29]:
# Add a normalized source_dataset label
df_securite["source_dataset"] = "securite_commune"

display(df_securite[["source_dataset"]].head())


,source_dataset
0,securite_commune
1,securite_commune
2,securite_commune
3,securite_commune
4,securite_commune


In [30]:
# Reorder columns
df_securite = df_securite[
    [
        "code_departement",
        "code_commune",
        "code_insee_commune",
        "annee",
        "indicateur",
        "unite_de_compte",
        "nombre",
        "taux_pour_mille",
        "est_diffuse",
        "insee_pop",
        "insee_pop_millesime",
        "insee_log",
        "insee_log_millesime",
        "complement_info_nombre",
        "complement_info_taux",
        "source_dataset",
        "source_dataset_timestamp",
        "date_refresh"
    ]
]

display(df_securite.head())
print(df_securite.dtypes)


,code_departement,code_commune,code_insee_commune,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux,source_dataset,source_dataset_timestamp,date_refresh
0,69,001,69001,2016,Violences physiques intrafamiliales,Victime,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
1,69,001,69001,2016,Violences physiques hors cadre familial,Victime,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
2,69,001,69001,2016,Violences sexuelles,Victime,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
3,69,001,69001,2016,Vols avec armes,Infraction,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
4,69,001,69001,2016,Vols violents sans arme,Infraction,1.581633,0.317097,ndiff,358,2016-01-01,163,2016-01-01,1.581633,0.317097,securite_commune,2026-03-23 14:13:09.597346,2026-03-23


code_departement                    object
code_commune                        object
code_insee_commune                  object
annee                                Int64
indicateur                          object
unite_de_compte                     object
nombre                             float64
taux_pour_mille                    float64
est_diffuse                         object
insee_pop                            int64
insee_pop_millesime         datetime64[ns]
insee_log                            int64
insee_log_millesime         datetime64[ns]
complement_info_nombre             float64
complement_info_taux               float64
source_dataset                      object
source_dataset_timestamp    datetime64[ns]
date_refresh                datetime64[ns]
dtype: object


In [31]:
# Basic checks
print("Rows:", len(df_securite))
print("Unique communes:", df_securite["code_insee_commune"].nunique())
print("Unique years:", sorted(df_securite["annee"].dropna().unique().tolist()))
print("Unique indicators:", df_securite["indicateur"].nunique())


Rows: 37125
Unique communes: 275
Unique years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Unique indicators: 15


In [32]:
#  save
output_path = Path("../../../data/silver/securite_data_silver_clean.csv")
df_securite.to_csv(output_path, index=False, sep=";", encoding="utf-8")

print(f"Saved to: {output_path}")


Saved to: ../../../data/silver/securite_data_silver_clean.csv
